### RaschPy SLM worked example

This notebook works through a sample Rasch analysis of a simulated data set (1,000 persons, 20 items, 40% missing data), taking you through the relevant commands step by step, with notes before each cell. Relevant outputs will appear below each cell.

Import the modules and set the working directory (here called `my_working_directory`) to where you want to save your output files.

In [ ]:
import raschpy as rp
import os

os.chdir('my_working_directory')

**A note on data validation**

Every model constructor validates the item-response network automatically at instantiation (`validate=True` by default) and warns if it's disconnected — if there's no chain of persons and items linking every item to every other, the resulting item locations aren't on a common scale, even though `calibrate()` will still run without error. This is worth seeing happen once. Here we build a small, deliberately disconnected data set: two groups of persons who each only answer a disjoint set of items, with no item shared between the groups to link them:

In [ ]:
sim_disconnected = rp.SLM_Sim(no_of_items=10, no_of_persons=40, item_range=2, seed=99)
responses_disconnected = sim_disconnected.responses.copy()
responses_disconnected.iloc[:20, 5:] = float('nan')   # first 20 persons: only answer items 1-5
responses_disconnected.iloc[20:, :5] = float('nan')   # last 20 persons: only answer items 6-10

broken_slm = rp.SLM(responses_disconnected)   # raises a UserWarning

`connectivity_status` records the diagnosis, including which items ended up in which isolated sub-group:

In [ ]:
broken_slm.connectivity_status

The rest of this notebook uses a single, fully-connected simulated data set, so this warning won't come up again.

Simulate a data set. Passing `seed=42` makes the simulation fully reproducible — rerunning this notebook will always generate the same data set. 50 items, 1,000 persons, 40% missing data (so each person responds to ~30 items and each item has ~600 responses).

In [ ]:
sim = rp.SLM_Sim(no_of_items=50, no_of_persons=1000, missing=0.4, seed=42)
sim.responses.to_csv('slm_scores.csv')

If you have your own response data saved to a CSV file instead of simulating it, use `loadup_slm()` to load and validate it — it coerces values to 0/1 and flags anything else as missing. Demonstrated here by reloading the file we just saved:

In [ ]:
data, invalid_responses = rp.loadup_slm('slm_scores.csv')

Check the data - view the first two lines

In [ ]:
data.head(2)

Check for any invalid responses (not usable for estimation purposes and excluded)

In [ ]:
invalid_responses

Create an SLM object. Passing the simulation object `sim` directly (rather than the reloaded `data`) attaches the generating parameters under `slm.generating`, which lets us check parameter recovery further down. If you are using your own data, pass a DataFrame (e.g. `rp.SLM(data)`) instead.

In [ ]:
slm = rp.SLM(sim)

Generate item estimates. The `%%time` "magic function" returns the time taken to run the cell contents (algorithm run time in this case).

In [ ]:
%%time
slm.calibrate()

Check the item location estimates - view the first two items

In [ ]:
slm.items.head(2)

Since this is simulated data, we know the true generating item locations (`slm.generating.items`) and can check how closely the calibration recovered them. The helper below plots generating vs. estimated values, with an identity line (dashed dark red) and a fitted regression line (dashed red) — the closer the points hug the identity line, the better the recovery. Also displays the Pearson correlation, SD ratio, regression coefficient and RMSE:

In [ ]:
import numpy as np
from matplotlib import pyplot as plt

def recovery_plot(generating, estimated, label, filename):
    x, y = np.asarray(generating), np.asarray(estimated)
    fig, ax = plt.subplots()
    ax.scatter(x, y, alpha=0.6)
    lo, hi = min(x.min(), y.min()), max(x.max(), y.max())
    ax.plot([lo, hi], [lo, hi], color='DarkRed', label='Identity')
    m, b = np.polyfit(x, y, 1)
    ax.plot([lo, hi], [m * lo + b, m * hi + b], color='Red', linestyle='--', label='Regression')
    ax.set_xlabel(f'Generating {label}')
    ax.set_ylabel(f'Estimated {label}')
    ax.set_aspect('equal')
    ax.legend()
    plt.savefig(filename)
    plt.show()
    print(f'{label} Pearson correlation:               {round(np.corrcoef(x, y)[0, 1], 3)}')
    print(f'{label} SD ratio (estimated / generating): {round(y.std() / x.std(), 3)}')
    print(f'{label} regression coefficient:            {round(m, 3)}')
    print(f'{label} RMSE:                              {round(np.sqrt(((x - y) ** 2).mean()), 3)}')

recovery_plot(slm.generating.items, slm.items, 'item location', 'my_slm_item_recovery.png')

Generate a table of item statistics (and check run time), and save to file

In [ ]:
%%time
slm.item_stats_df(full=True)
slm.item_stats.to_csv('slm_item_stats.csv')

Check the item statistics table - view the first ten items with `.head(10)`

In [ ]:
slm.item_stats.head(10)

Generate a table of person statistics (and check run time), and save to file

In [ ]:
%%time
slm.person_stats_df(full=True)
slm.person_stats.to_csv('slm_person_stats.csv')

Check the person statistics table - view the first ten persons with `.head(10)`

In [ ]:
slm.person_stats.head(10)

And the same recovery check for person locations, against `slm.generating.persons`:

In [ ]:
recovery_plot(slm.generating.persons, slm.persons, 'person location', 'my_slm_person_recovery.png')

Generate a table of test-level statistics (and check run time), and save to file

In [ ]:
%%time
slm.test_stats_df()
slm.test_stats.to_csv('slm_test_stats.csv')

Check the test statistics table

In [ ]:
slm.test_stats

Run a residual correlation analysis (and check run time), and save relevant output to file

In [ ]:
%%time
slm.res_corr_analysis()
slm.residual_correlations.to_csv('slm_residual_correlations.csv')
slm.loadings.to_csv('slm_loadings.csv')

View the first 10 rows of the table of pairwise standard residual correlations using .head(10)

In [ ]:
round(slm.residual_correlations.head(10), 3)

View the first 10 item loadings on the first principal component of the pairwise standard residual correlations (dimensionality test)

In [ ]:
round(slm.loadings['PC 1'].head(10), 3)

Produce an item characteristic curve (item response function) curve for Item 2, with observed category means plotted and the threshold (item location) marked

In [ ]:
slm.icc('Item_2', title='ICC for Item 2', obs=True, thresh_line=True, xmin=-5, xmax=3, filename='my_slm_icc')

Produce an item information function curve for Item 2

In [ ]:
slm.iic('Item_2', point_info_lines=[0], point_info_labels=True, title='Information for Item 2',
        xmin=-5, xmax=4, filename='my_slm_iic')

Produce a test characteristic curve (test response function), with person locations corresponding to scores of 10 and 15 plotted.

In [ ]:
slm.tcc(score_lines=[20, 35], obs=True, score_labels=True, filename='my_slm_tcc')

Produce a test information curve

In [ ]:
slm.test_info(point_info_lines=[0], point_info_labels=True, filename='my_slm_test_info_curve')

Produce a test CSEM (conditional standard error of measurement) curve, with the CSEM corresponding to a person location of -3 plotted

In [ ]:
slm.test_csem(point_csem_lines=[-3], point_csem_labels=True, ymax=2.5, filename='my_slm_csem_curve')

Produce a histogram of standardised residuals, with a normal distribution curve overlaid

In [ ]:
slm.std_residuals_plot(bin_width=0.6, normal=True, filename='my_slm_std_residuals_plot')